# FarmTech Solutions — Fase 6 revisada
Enzo Caetano Peracio Rodrigues — RM 570352

Relatório dos experimentos reais de 13/10/2026 (Colab Tesla T4).
Esta revisão organiza o material já executado; não representa novos treinamentos.
Código de infraestrutura e reprodução: [notebook auxiliar](reproducao.ipynb).
O [original com todas as saídas](original.ipynb) foi preservado sem alteração.


## 1. Objetivo, dados e método
80 imagens, 40 por classe (bottle/backpack), 32/4/4 por classe.
Comparação: YOLOv5s do zero (30/60 épocas), YOLO COCO e CNN própria.
A seleção usa validação; as abordagens compartilham as oito imagens de teste.
O experimento complementar com pesos COCO é apresentado separadamente.
Veja [procedência e Make Sense](../../docs/visao/makesense_validation.md).


In [1]:
from pathlib import Path
import json
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "experiments/objetos/results").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Abra o notebook dentro do repositório FarmTech")
RESULTS = ROOT / "experiments/objetos/results"

def show_json(relative):
    data = json.loads((RESULTS / relative).read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, ensure_ascii=False))


## 2. YOLO do zero — comparação concluída (antiga seção 9.3)
| Experimento | Melhor época | mAP50 validação | mAP50:95 validação | Treino (s) |
|---|---:|---:|---:|---:|
| 30 épocas | 30 | 0,00096376 | 0,000096376 | 165,22 |
| 60 épocas | 59 | 0,13642 | 0,040407 | 294,21 |

A validação selecionou 60 épocas; o teste não participou da escolha.
O ganho na validação não se repetiu no teste.


In [2]:
show_json('yolo_custom/test_metrics.json')

{
  "precision": 0.0020524113661964686,
  "recall": 0.525,
  "map50": 0.013059191143228482,
  "map50_95": 0.005039173957851236,
  "box_loss": 0.0587373748421669,
  "obj_loss": 0.014763751067221165,
  "cls_loss": 0.01799142174422741,
  "metrics_split": "test",
  "loss_computed": true,
  "evaluation_precision": "FP32",
  "images_evaluated": 8,
  "count": 8,
  "mean_inference_ms": 34.22255749995884,
  "median_inference_ms": 35.99341850008386,
  "std_inference_ms": 6.733129515945234,
  "dataset_fingerprint": "f8bb47c02934ae7bf3b4d2256eed4ec062bec252e1f6411d1cfecc90f67d3cc1",
  "environment": {
    "python": "3.12.13",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
    "torch": "2.5.1+cu124",
    "tensorflow": "2.17.1",
    "cuda": "12.4",
    "gpu": "Tesla T4",
    "tensorflow_gpu_count": 1,
    "gpu_smoke_test": "torch operation and TensorFlow convolution passed",
    "torch_cudnn": 90100,
    "tensorflow_build": {
      "cpu_compiler": "/usr/lib/llvm-17/bin/clang",
      "cuda_c

## 3. YOLO COCO e CNN própria
A YOLO COCO original detectou garrafas, sem demonstrar mochilas a confiança 0,25.
A CNN classificou corretamente 7/8 imagens (accuracy 0,875; F1 macro 0,873016).
Matriz de confusão: [[4,0],[1,3]]. Detecção e classificação têm métricas distintas.


In [3]:
show_json('cnn/metrics.json')

{
  "accuracy": 0.875,
  "precision": 0.9,
  "recall": 0.875,
  "f1": 0.873015873015873,
  "averaging": "macro",
  "confusion_matrix": [
    [
      4,
      0
    ],
    [
      1,
      3
    ]
  ],
  "class_names": [
    "bottle",
    "backpack"
  ],
  "metrics_split": "test",
  "dataset_fingerprint": "f8bb47c02934ae7bf3b4d2256eed4ec062bec252e1f6411d1cfecc90f67d3cc1",
  "environment": {
    "python": "3.12.13",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
    "torch": "2.5.1+cu124",
    "tensorflow": "2.17.1",
    "cuda": "12.4",
    "gpu": "Tesla T4",
    "tensorflow_gpu_count": 1,
    "gpu_smoke_test": "torch operation and TensorFlow convolution passed",
    "torch_cudnn": 90100,
    "tensorflow_build": {
      "cpu_compiler": "/usr/lib/llvm-17/bin/clang",
      "cuda_compute_capabilities": [
        "sm_60",
        "sm_70",
        "sm_80",
        "sm_89",
        "compute_90"
      ],
      "cuda_version": "12.3",
      "cudnn_version": "8",
      "is_cuda_build": t

## 4. Comparação das abordagens
| Abordagem | Métrica principal de teste | Média / mediana de inferência (ms) |
|---|---|---|
| YOLO própria | mAP50:95 = 0,005039 | 34,22 / 35,99 |
| YOLO COCO | mAP indisponível neste baseline | 20,97 / 20,22 |
| CNN própria | accuracy = 0,875 | 16,99 / 16,30 |

Confiança não é precisão. Os tempos pertencem a pipelines e entradas diferentes,
com teste pequeno; não constituem um ranking universal.


## 5. Experimento complementar com pesos COCO
30/60 épocas com os mesmos dados. A validação selecionou 60 épocas,
checkpoint da época 58. O teste observado demonstrou as duas classes:
mAP50 = 0,9950; mAP50:95 = 0,759696.
É avaliação complementar retrospectiva, não novo teste cego.


In [4]:
show_json('yolo_improved/test_metrics.json')

{
  "precision": 0.8204680922958334,
  "recall": 0.9799404396427691,
  "map50": 0.995,
  "map50_95": 0.759696312620182,
  "box_loss": 0.016924338415265083,
  "obj_loss": 0.004972620867192745,
  "cls_loss": 0.0013422972988337278,
  "metrics_split": "test",
  "loss_computed": true,
  "evaluation_precision": "FP32",
  "images_evaluated": 8,
  "dataset_fingerprint": "f8bb47c02934ae7bf3b4d2256eed4ec062bec252e1f6411d1cfecc90f67d3cc1",
  "checkpoint_sha256": "6fedc77d0d511c5e86f9fb4a1a6abe8bdc52ef62b6e6e393e1c51b3c68082eb4",
  "initialization": "COCO pretrained; supplementary fine tuning",
  "evaluation_note": "Original test images were already examined in the preceding experiment; this is a retrospective supplementary evaluation, not a fresh blind test."
}


![Exemplos reais do teste complementar](../../experiments/objetos/results/yolo_improved/test/gallery.png)

## 6. Conclusões e limitações
Preservam-se os resultados fracos do treino do zero e a melhoria com transferência.
Oito imagens de teste limitam generalizações. Há falsos positivos e imagens sem
detecção; consulte [análise dos resultados](../../docs/visao/results_analysis.md).

O detector de tomate da Fase 7 utiliza outro dataset e outro experimento.
Nenhuma métrica deste relatório é atribuída à aplicação agrícola.
